# 해설서 추출하여 텍스트 파일로 저장

In [1]:
import requests

URL = 'https://unipass.customs.go.kr/clip/index.do'
response = requests.get(URL)
print(response.status_code)

200


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
import re


In [4]:
# Ryu = [str(i).zfill(2) for i in range(1, 98)]
# print(Ryu)

In [7]:
Ryu = [str(i).zfill(2) for i in range(78, 98)]

for n in Ryu:
    total_explanation = {}         # 모든 HS 품목 류 담을 딕셔너리
    sub_explanation = {}           # 특정 류, 호 담을 딕셔리리
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=webdriver.ChromeOptions())
    driver.get('https://unipass.customs.go.kr/clip/index.do')
    ele = driver.find_element(by=By.ID, value='TOPMENU_LNK_M_ULS0200000000')
    ele.send_keys(Keys.ENTER)
    time.sleep(3)

    
    ele = driver.find_element(By.ID,'searchVal')
    ele.send_keys(n)
    ele.send_keys(Keys.ENTER)
    time.sleep(3)
        
    
    # 류 해설서
    ryu = driver.find_element(by=By.CSS_SELECTOR, value='#divLft_tab3 > pre').text
    # ryu = re.sub('[※ㅇ\-•●◆▶★]', '', ryu)                      # 특수기호 제거
    ryu = re.sub(r'\(([\u4E00-\u9FFF]{1,10})\)', '', ryu)      # 한자 제거
    # ryu = ryu.replace('\n', ' ')                               # 개행문자 제거
    sub_explanation['ryu'] = ryu
    
    # 호 해설서
    ho_ex_dict = {}
    cnt = len(driver.find_elements(by=By.CSS_SELECTOR, value='#tblLstBody > tr'))
    for i in range(1,cnt+1):
        value = '#tblLstBody > tr:nth-child(%d) > td:nth-child(2) > a' %i
        ho = driver.find_element(by=By.CSS_SELECTOR, value=value)
        ho.click()
        time.sleep(5)
        ho_ex = driver.find_element(by=By.CSS_SELECTOR, value='#divLft_tab4 > pre').text
        # ho_ex = re.sub('[※ㅇ\-•●◆▶★]', '', ho_ex)                      # 특수기호 제거
        ho_ex = re.sub(r'\(([\u4E00-\u9FFF]{1,10})\)', '', ho_ex)      # 한자 제거
        # ho_ex = ho_ex.replace('\n', ' ')                               # 개행문자 제거
        ho_ex_dict[i] = ho_ex
        driver.back()
        time.sleep(5)
    sub_explanation['ho'] = ho_ex_dict
    
    
    # 텍스트 파일로 저장하기
    with open('./HS_explanation_22.txt', 'w', encoding='utf-8') as f:
        f.write(str(sub_explanation))
    
    # 22류 해설서 추출하여 csv 파일로 저장
    import pandas as pd
    
    df_22 = pd.DataFrame(sub_explanation)
    
    # 류 번호, 류 해설, 호 번호, 호 해설 분리하기
    df_22['ryu_ex'] = df_22['ryu'].str[5:]   # 류 해설
    df_22['ryu'] = df_22['ryu'].str[1:3]     # 류 번호
    df_22['ho_ex'] = df_22['ho'].str[6:]     # 호 해설
    df_22['ho'] = df_22['ho'].str[3:5]       # 호 번호
    
    
    # HS_4 컬럼 추가
    df_22['HS_4'] = df_22['ryu'] + df_22['ho']
    
    # 컬럼 정렬
    df_22 = df_22.reindex(columns=['HS_4', 'ryu', 'ryu_ex', 'ho', 'ho_ex'])
    
    # csv 파일 저장하기
    df_22.to_csv(f'./해설서/HS_{n}_explanation.csv', index=False)

    # 창닫기 
    driver.close()
    
